The goal of this penultimate notebook is to choose the best model for data derived in nb7. It deals exclusively with model architecture and fine tuning.

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import joblib
import xgboost as xgb
import matplotlib.pyplot as plt

from helpers import preprocessing, submissions, preprocessing

from helpers.utils import (
    stats_helper,
    plotting_helper,
    unsupervised_helper,
    aggregator_helper,
    sklearn_helper,
)
import lightgbm
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier


from helpers.model_builders import bureau_preparation

from helpers.utils.cleaner_helper import plot_categorical_dist, plot_numeric_dist

from lightgbm import LGBMClassifier
import hdbscan
import importlib

In [2]:
data = joblib.load("data/processed/all_tables.jbl")
data.info()

X_train = data[~data["TARGET"].isna()]
X_train = X_train.drop(columns=["SK_ID_CURR"]).reset_index(drop=True)
y_train = X_train.pop("TARGET")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 356255 entries, 0 to 356254
Columns: 473 entries, SK_ID_CURR to onehot_count_NAME_CONTRACT_STATUS_S_y
dtypes: bool(34), category(14), float64(417), int64(7), object(1)
memory usage: 1.1+ GB


In [3]:
importlib.reload(sklearn_helper)
def val_model(model, X_train = X_train) -> pd.DataFrame:
    return sklearn_helper.stratified_cv_model(
        model,
        X_train,
        y_train,
        scoring=['average_precision', 'roc_auc', 'f1_macro'])

# Dedicated model libraries
## LightGBM

#### Gradient Boosted Tree

In [4]:
val_model(LGBMClassifier(boosting_type='gbdt', verbose=-1))

,average_precision,roc_auc,f1_macro
mean,0.2691,0.7784,0.5143
std,0.0055,0.0043,0.0007


#### Random_forest

In [5]:
val_model(LGBMClassifier(boosting_type='rf', verbose=-1,
        bagging_fraction=0.2, bagging_freq=5))

,average_precision,roc_auc,f1_macro
mean,0.2113,0.7217,0.5545
std,0.0051,0.0043,0.0023


The parameters used, make the RF significantly underperform when compared to gradient boosted trees.
* F1 macro score is improved only due to class imbalance.

#### DART

In [6]:
val_model(LGBMClassifier(boosting_type='dart', verbose=-1))

,average_precision,roc_auc,f1_macro
mean,0.2568,0.7659,0.4857
std,0.0066,0.0038,0.0010


Performance slightly worse in all regards, when compared to gradient boosted trees.

## XGBoost
XGboost provides alternative gradient boosted tree application. More importantly, it has been shown to outperform LightGBM for large datasets.

In [7]:
model = xgb.XGBClassifier(
    objective="binary:logistic",
    enable_categorical=True,
)

val_model(model)

,average_precision,roc_auc,f1_macro
mean,0.2496,0.7660,0.5357
std,0.0051,0.0025,0.0038


In [8]:
model = xgb.XGBClassifier(
    objective="reg:logistic",
    enable_categorical=True,
)

val_model(model)

,average_precision,roc_auc,f1_macro
mean,0.2496,0.7660,0.5357
std,0.0051,0.0025,0.0038


# Nan-sensitive models
Models unable to impute nan values will need to rely on an imputer. 

## NAN classification
As data imputation will be introducing noise and bias, it is important to first quantify how much data is missing and whether features need to be discarded.

Models of different architecture are tested with the goal of building an ensemble. 

In [ ]:
num_cols = X_train.select_dtypes(include=[np.number]).columns.tolist()
num_cols = [col for col in num_cols if X_train[col].notna().sum() > 0]


cat_cols = X_train.select_dtypes(include=["category"]).columns.tolist()
cat_cols = [col for col in cat_cols if X_train[col].notna().sum() > 0]


from helpers.utils import sklearn_helper


cat_as_obj_transformer = make_column_transformer(
    (FunctionTransformer(lambda x: x.astype("str")), cat_cols),
    remainder="passthrough",
    verbose_feature_names_out=False,
)
cat_as_obj_transformer.set_output(transform="pandas")

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(
                [
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            num_cols,
        ),
        (
            "cat",
            Pipeline(
                [
                    ("str_converter", sklearn_helper.category_transformer(cat_cols, "str")),
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]
            ),
            cat_cols,
        ),
    ],
    remainder="drop",
)
display(preprocessor)
X_imputed = preprocessor.fit_transform(X_train[0:10000])
X_imputed.shape

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


c:\GITHUB\Turing submissions\psauci-DS.v2.5.3.4.1\.venv\Lib\site-packages\sklearn\impute\_base.py:637: UserWarning: Skipping features without any observed values: ['max_credit_usage_Other' 'median_credit_usage_Other'
 'min_credit_usage_Other']. At least one non-missing value is needed for imputation with strategy='median'.
  warnings.warn(


(10000, 548)

This preprocessor is then tested on 4 unique structure sklearn models capable of outputting ROC_AUC curves.

In [40]:
sk_models = [
    LogisticRegression(max_iter=1000, random_state=3),
    RandomForestClassifier(random_state=3),
    KNeighborsClassifier(),
    HistGradientBoostingClassifier(random_state=3),
]

sk_pipes = [
  
    Pipeline([("preprocessor", preprocessor), (type(model).__name__, model)])
    for model in sk_models]


display(sk_pipes[0])

,steps,"[('preprocessor', ...), ('LogisticRegression', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [ ]:
for model in sk_pipes:
    print(f"Validating: {model[-1]}")
    try:
        display(val_model(model, X_train))
    except Exception as e:
        print(f"Error validating {type(model)}: {e}")

Validating: LogisticRegression(max_iter=1000, random_state=3)


## Voting Classifier
The best performing models are then investigated along their ROC and AUC curves.

In [ ]:
voter = VotingClassifier(
    estimators=[
        ("lgbm_gbdt", LGBMClassifier(boosting_type='gbdt', verbose=-1)),
        sk_pipes,
    ],
    voting="soft",
    n_jobs=-1,
)

display(voter)

SyntaxError: invalid syntax (1346657977.py, line 10)

In [ ]:
eval_model(voter)